In [200]:
import pandas as pd
import numpy as np
import re


In [ ]:
months_pattern = "(january|february|march|april|may|june|july|august|september|october|november|december|jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec)"

#Checks if a column is a period column. 
def is_period_column(col):
    if not isinstance(col, str):
        return False
        
    patterns = [
        r"FY\d{2,4}",      # FY2024, FY24
        r"\b\d{4}\b",      # 2024
        r"[A-Za-z]{3,9}\s?\d{2,4}",  # Jun 2024
        months_pattern + r"[a-z]*"
        # r"(jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec)[a-z]*",
        # r"(jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec)[a-z]*",
    ]
    
    return any(re.search(p, col, re.IGNORECASE) for p in patterns)

def extract_period_unit(col):
    col_clean = col.replace("_", " ") # Temporarily replace underscores with spaces for easier matching
    
    # Pattern to grab "30 June 2025" or "June 2025"
    date_pattern = r"(\d{1,2}\s+)?" + months_pattern + r"(\s*\d{2,4})?"
    # date_pattern = r"(\d{1,2}\s+)?(january|february|march|april|may|june|july|august|september|october|november|december|jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec)\s+\d{2,4}"
    
    period_match = re.search(date_pattern, col_clean, re.IGNORECASE)
    # If no full date, fall back to FY or Year
    if not period_match:
        period_match = re.search(r"(fy\d{2,4}|\b\d{4}\b)", col_clean, re.IGNORECASE)
        
    # unit_match = re.search(r"\$m|%", col_clean) # Specifically looking for $m or % based on your data
    unit_match = re.search(r"\$'?[\w%]*|%+|bps", col_clean)

    # category_match = re.search()

    period = period_match.group(0) if period_match else col
    unit = unit_match.group(0) if unit_match else None
    
    return period, unit

# 7️⃣ Clean metric labels (remove units)
def clean_metric(text):

    if pd.isna(text):
        return text

    text = str(text).lower()

    # remove units like ($m)
    text = re.sub(r"\(.*?\)", "", text)

    # remove trailing footnote numbers
    text = re.sub(r"\s+\d+(,?\d)*", "", text)

    # remove superscript footnotes
    text = re.sub(r"[¹²³⁴⁵⁶⁷⁸⁹⁰]", "", text)

    # remove star footnotes
    text = re.sub(r"\*", "", text)

    text = text.strip()

    return text

# 8️⃣ Cleans datapoints to ensure that there are no footnotes etc. 
def clean_number(x):

    if pd.isna(x):
        return x

    x = str(x).strip()

    # detect negative via parentheses
    is_negative = bool(re.search(r"\(\s*\d+\.?\d*", x))

    # remove commas
    x = x.replace(",", "")

    # remove footnote markers
    x = re.sub(r"[¹²³⁴⁵⁶⁷⁸⁹⁰\*]", "", x)

    # extract all numbers
    numbers = re.findall(r"\d+\.?\d*", x)

    if numbers:
        value = float(numbers[0])

        if is_negative:
            value = -value

        # percentage handling
        if "%" in x:
            return value / 100

        return value

    return x 

def is_unit_column(df, col_name):
    # 1. Standardize the header name
    header = str(col_name).lower().strip()
    
    # Check if header explicitly mentions units
    unit_indicators = [r'\b\$\b', r'\b%\b', r'\bbps\b', r'\bunit\b', r'currency', r'multiplier']
    if any(re.search(p, header) for p in unit_indicators):
        if "unnamed" not in header: # Avoid false positives on default pandas names
            return True

    # 2. Check the column content (Crucial for empty/NaN headers)
    # Grab the first few non-null values
    sample_values = df[col_name].dropna().head(5).astype(str).str.lower().str.strip()
    
    if sample_values.empty:
        return False
        
    # Check if the values are strictly unit markers (e.g., "$m", "%", "bps")
    # This prevents matching actual data columns that happen to have one symbol

    FINANCIAL_UNITS = {
        # Symbols
        '$', '%', '(', ')', '#','number'
        # Scalers
        '$m', 'm', 'mn', 'million', 'millions',
        '$b', 'b', 'bn', 'billion', 'billions',
        '$k', 'k', '000', '000s',
        # Rates
        'bps', 'bp', 'pct', 'percent',
        'times','year','years','month','months'
        # Currency units
        'c', 'cps', 'cpu', 'cents', 'usd', 'aud', 'eur','sgd','jpy','cents per share','cents per unit'
    }

    are_all_units = sample_values.apply(lambda x: x in FINANCIAL_UNITS).all()
    
    return are_all_units

In [239]:


def clean_financial_table(df):

    #Create a copy of the original dataframe. 
    df = df.copy()

    # 1️⃣ Replace blank strings with NaN
    df = df.replace(r'^\s*$', np.nan, regex=True)

    # 2️⃣ Drop completely empty rows and columns
    df = df.dropna(axis=0, how="all")
    df = df.dropna(axis=1, how="all")

    # 3️⃣ Reset index
    df = df.reset_index(drop=True)

    # 4️⃣ Detect header row
    header_row = None

    for i in range(min(3, len(df))):

        row_text = " ".join(df.iloc[i].astype(str).str.lower())

        # header_patterns = r"(?i)\b(jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec|fy|hy|q[1-4]|20\d{2})\b"
        header_patterns = r"(?i)\b(jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec|january|february|march|april|may|june|july|august|september|october|november|december)\b|fy|hy|q[1-4]|20\d{2}"
        # r'(fy|hy|FY|HY|20\d{2}|q[1-4])'
        if re.search(header_patterns, row_text):
            header_row = i
            break

    # Promote header if detected
    if header_row is not None:

        df.columns = df.iloc[header_row]
        df = df.drop(header_row).reset_index(drop=True)

    else:
        df.columns = [f"col_{i}" for i in range(len(df.columns))]



    # 5️⃣ Normalize column names
    df.columns = (
        pd.Series(df.columns)
        .astype(str)
        .str.lower()
        .str.strip()
        .str.replace(r"\s+", "_", regex=True)
    )

    # 6️⃣ Ensure first column is metric column
    df = df.rename(columns={df.columns[0]: "metric"})
    

    #create unit column if it exist
    # 1. Identify which column is the unit column
    unit_cols = [col for col in df.columns if is_unit_column(df, col)]
    print("unitcols",unit_cols)


    if unit_cols:
        # Rename the first detected unit column to exactly 'unit'
        df = df.rename(columns={unit_cols[0]: 'unit'})
        print(f"Renamed {unit_cols[0]} to 'unit'")

    
    #This removes unnecessary columns 
    period_cols = [c for c in df.columns if is_period_column(str(c))]
    df = df[["metric"] + ([ "unit" ] if "unit" in df.columns else []) + period_cols]
    

    #Code here stores units and period into separate dictionaries. 
    period_map = {}
    unit_map = {}

    for col in period_cols:
        period, unit = extract_period_unit(col)
        period_map[col] = period
        unit_map[col] = unit
        
    #Cleans only for metric column.
    df['metric'] = df['metric'].map(clean_metric)

    #Apply clean_number_conditionally only to non metric cols. 
    cols_excluding_metric = df.columns.difference(['metric'])
    df[cols_excluding_metric] = df[cols_excluding_metric].map(clean_number)

    # Check if 'unit' exists in columns before melting
    id_variables = ["metric"]

    
    if 'unit' in df.columns:
        id_variables.append("unit")

    df_long = df.melt(
        id_vars = id_variables,
        var_name = "column",
        value_name = "value"
    )

    

    df_long["period"] = df_long["column"].map(period_map)

    if 'unit' not in df_long.columns:
        df_long["unit"] = df_long["column"].map(unit_map) 

    return df_long

In [243]:
test_table = r"datasets\processed\tables\CIP\FY25\CIP_FY25_IP-table-2"
# test_table = r"datasets\processed\tables\ABG\FY17\ABG_FY17_IP-table-5"

df = pd.read_csv(test_table,index_col=0,header=None)

In [244]:
df

,1,2,3
0,,,
NaN,Portfolio snapshot,NaN,FY25 1
0.0,Number of assets,#,87
1.0,Book value,$m,"3,890"
2.0,WACR,%,5.86
3.0,GLA,sqm,"1,293,790"
4.0,Average asset size,sqm,"14,871"
5.0,Average tenancy size,sqm,"7,610"
6.0,Occupancy by income 2,%,95.1
7.0,WALE by income 2,years,7.1


In [245]:
clean_financial_table(df=df)

unitcols []


,metric,column,value,period,unit
0,number of assets,fy25_1,87.00,fy25,None
1,book value,fy25_1,3890.00,fy25,None
2,wacr,fy25_1,5.86,fy25,None
3,gla,fy25_1,1293790.00,fy25,None
4,average asset size,fy25_1,14871.00,fy25,None
5,average tenancy size,fy25_1,7610.00,fy25,None
6,occupancy by income,fy25_1,95.10,fy25,None
7,wale by income,fy25_1,7.10,fy25,None
8,landholding,fy25_1,296.00,fy25,None
9,freehold ownership,fy25_1,99.00,fy25,None
